# Vetorizar um livro de gramática — PDF selecionável

Este notebook prepara um acervo citável com E5 Large. É independente do
aplicativo: não altera o Língua nem o Agente Analista e não faz análise
sintática do relato. A importação do acervo no aplicativo será implementada depois.

No Colab, selecione **Ambiente de execução → Alterar tipo de ambiente de
execução → GPU**. O alvo é A100; a GPU e sua VRAM reais serão verificadas.
Os 80 GB da GPU não são a RAM da máquina.

Execute por etapas. Há uma pausa para conferir a extração antes dos vetores.
Ao enviar o PDF ao Colab, ele será processado na infraestrutura do Google.
O modelo é baixado do Hugging Face; não são usados serviços de embeddings.
A retomada depende de manter a pasta de trabalho; use Drive para persistência
entre sessões. O PDF e os pesos não são incluídos no ZIP final.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import textwrap
import venv

TITULO = "Nova Gramática do Português Contemporâneo"
AUTOR = "Celso Cunha; Lindley Cintra"
EDICAO = "7ª edição, 2ª impressão"
ANO = "2017"
CAMINHO_PDF = ""  # Deixe vazio para escolher um PDF do computador na etapa 3.
USAR_DRIVE = True  # Preserva trabalho e checkpoints entre sessões do Colab.
PASTA_BASE = Path("/content/gramatica_livro")
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_BASE = Path("/content/drive/MyDrive/gramatica_livro")
PASTA_BASE.mkdir(parents=True, exist_ok=True)

ALVO_TOKENS = 380
LIMITE_TOKENS = 490   # Inclui título, prefixo e tokens especiais; modelo aceita até 512.
LOTE_INICIAL = 128    # Reduzido automaticamente em falta de memória na GPU.
EXTRACAO_CONFERIDA = False
FRAGMENTOS_CONFERIDOS = False
# Recorte previamente usado: só é aplicado ao PDF reconhecido pelo SHA-256.
USAR_CORPO_CUNHA_RECONHECIDO = True
PAGINAS_INCLUIR = None  # None = automático; ou lista de páginas PDF (começa em 1).

In [ ]:
FONTES = {'gramatica_pdf.py': '"""Extração, preparação revisável e fragmentação de um PDF textual para E5.\n\nNão executa OCR nem corrige a ordem espacial do PDF. O texto extraído permanece\nintacto; exclusões e recomposições pertencem a uma camada derivada, com trilha\nde origem. As posições são índices de caracteres Python, não bytes do PDF.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport copy\nimport json\nimport math\nimport re\nimport statistics\nfrom pathlib import Path\n\n\nclass ErroPreparacaoLivro(ValueError):\n    """Um arquivo ou um limite não permite uma preparação sem perda silenciosa."""\n\n\ndef _sha256(texto):\n    return hashlib.sha256(texto.encode("utf-8")).hexdigest()\n\n\ndef _aviso(codigo, mensagem, **detalhes):\n    return {"codigo": codigo, "mensagem": mensagem, **detalhes}\n\n\ndef _numero(valor):\n    return isinstance(valor, (int, float)) and not isinstance(valor, bool) and math.isfinite(valor)\n\n\ndef _colunas_ou_ordem_ambigua(blocos):\n    """Um alerta conservador; não constitui um detector de layout de leitura."""\n    caixas = [b["bbox"] for b in blocos if b["texto"].strip()]\n    for i, a in enumerate(caixas):\n        for b in caixas[i + 1:]:\n            altura_comum = min(a[3], b[3]) - max(a[1], b[1])\n            separados = a[2] + 8 < b[0] or b[2] + 8 < a[0]\n            if altura_comum > 4 and separados:\n                return True\n    return any(b[1] + 12 < a[1] for a, b in zip(caixas, caixas[1:]))\n\n\ndef _possivel_tabela(linhas):\n    """Três faixas de altura com três células: alerta, nunca remoção."""\n    faixas = []\n    for linha in sorted(linhas, key=lambda l: l["bbox"][1]):\n        caixa = linha["bbox"]\n        faixa = next((f for f in faixas if abs(f["y"] - caixa[1]) <= 3), None)\n        if faixa is None:\n            faixa = {"y": caixa[1], "caixas": []}\n            faixas.append(faixa)\n        faixa["caixas"].append(caixa)\n    return sum(len(f["caixas"]) >= 3 for f in faixas) >= 3\n\n\ndef extrair_pdf(caminho, metadados=None):\n    """Extrai blocos na ordem declarada pelo PDF, com páginas e offsets.\n\n    ``metadados`` permite registrar título, autoria e edição informados pelo\n    usuário. Eles ficam separados dos metadados declarados no próprio PDF.\n    ``pagina_rotulo`` vem do catálogo do PDF; não é inferida como página impressa.\n    """\n    try:\n        import pymupdf as fitz\n    except ImportError:\n        try:\n            import fitz\n        except ImportError as erro:\n            raise ErroPreparacaoLivro("Instale PyMuPDF para extrair o PDF.") from erro\n\n    caminho = Path(caminho)\n    if not caminho.is_file():\n        raise ErroPreparacaoLivro("O arquivo PDF não foi encontrado.")\n    if metadados is not None and not isinstance(metadados, dict):\n        raise ErroPreparacaoLivro("Os metadados bibliográficos devem ser um objeto.")\n    metadados = json.loads(json.dumps(metadados or {}, ensure_ascii=False, allow_nan=False))\n    digest = hashlib.sha256()\n    with caminho.open("rb") as arquivo:\n        for parte in iter(lambda: arquivo.read(1024 * 1024), b""):\n            digest.update(parte)\n    avisos, paginas, sumario = [], [], []\n    try:\n        documento = fitz.open(str(caminho))\n    except Exception as erro:\n        raise ErroPreparacaoLivro("Não foi possível abrir o PDF: " + str(erro)) from erro\n    with documento:\n        if not documento.is_pdf:\n            raise ErroPreparacaoLivro("O arquivo deve ser um PDF.")\n        if documento.needs_pass:\n            raise ErroPreparacaoLivro("O PDF está protegido por senha; use uma cópia acessível.")\n        if not len(documento):\n            raise ErroPreparacaoLivro("O PDF não contém páginas.")\n        metadata_pdf = dict(documento.metadata or {})\n        for item in documento.get_toc(simple=True):\n            if len(item) >= 3 and isinstance(item[2], int) and 1 <= item[2] <= len(documento):\n                sumario.append({"nivel": item[0], "titulo": item[1], "pagina_pdf": item[2]})\n        if not sumario:\n            avisos.append(_aviso("sem_sumario_pdf", "O PDF não contém marcadores de seção; nenhum título foi inferido."))\n        for indice, pagina in enumerate(documento):\n            numero = indice + 1\n            blocos, partes, cursor = [], [], 0\n            estrutura = pagina.get_text("dict", sort=False)\n            estilos = {}\n            for grupo in estrutura.get("blocks", []):\n                if grupo.get("type") != 0:\n                    continue\n                spans = [s for linha in grupo.get("lines", []) for s in linha.get("spans", [])]\n                tamanhos = [float(s["size"]) for s in spans if s.get("text", "").strip()]\n                if tamanhos:\n                    estilos[tuple(round(float(v), 2) for v in grupo["bbox"])] = {\n                        "tamanho_maximo": max(tamanhos), "tamanho_mediano": statistics.median(tamanhos),\n                        "linhas": len(grupo.get("lines", [])),\n                        "tem_negrito": any(int(s.get("flags", 0)) & 16 for s in spans),\n                    }\n            for bloco in pagina.get_text("blocks", sort=False):\n                if len(bloco) < 7 or bloco[6] != 0:\n                    continue\n                texto = bloco[4]\n                if not isinstance(texto, str) or not texto:\n                    continue\n                if partes:\n                    partes.append("\\n\\n")\n                    cursor += 2\n                bbox = [float(v) for v in bloco[:4]]\n                if not all(_numero(v) for v in bbox):\n                    raise ErroPreparacaoLivro(f"Coordenadas inválidas na página {numero}.")\n                blocos.append({\n                    "id": f"pagina-{numero}:bloco-{len(blocos)}", "ordem": len(blocos),\n                    "texto": texto, "bbox": bbox, "offset_inicio": cursor,\n                    "offset_fim": cursor + len(texto),\n                    "estilo": estilos.get(tuple(round(v, 2) for v in bbox), {}),\n                })\n                partes.append(texto)\n                cursor += len(texto)\n            texto_pagina = "".join(partes)\n            caracteres = sum(not c.isspace() for c in texto_pagina)\n            if not caracteres:\n                avisos.append(_aviso("pagina_sem_texto", "Página sem texto extraível; não foi aplicado OCR.", pagina_pdf=numero))\n            elif caracteres < 40 or texto_pagina.count("\\ufffd") > max(2, caracteres // 100):\n                avisos.append(_aviso("extracao_pobre", "Pouco texto ou caracteres de substituição: conferir esta página.", pagina_pdf=numero))\n            # O extrator pode reunir duas colunas no mesmo bloco. As caixas\n            # de linhas permitem sinalizar isso sem reordenar o texto citado.\n            linhas = [{"bbox": linha["bbox"], "texto": "linha"}\n                      for grupo in estrutura.get("blocks", [])\n                      if grupo.get("type") == 0 for linha in grupo.get("lines", [])]\n            if _colunas_ou_ordem_ambigua(blocos) or _colunas_ou_ordem_ambigua(linhas):\n                avisos.append(_aviso("ordem_leitura_ambigua", "Possíveis colunas ou ordem de blocos ambígua; a ordem do PDF foi preservada.", pagina_pdf=numero))\n            if _possivel_tabela(linhas):\n                avisos.append(_aviso("possivel_quadro_ou_tabela", "Células alinhadas podem formar quadro ou tabela; a relação espacial exige conferência no PDF. Nenhum conteúdo foi removido.", pagina_pdf=numero))\n            paginas.append({\n                "pagina_pdf": numero, "pagina_rotulo": pagina.get_label() or None,\n                "largura": float(pagina.rect.width), "altura": float(pagina.rect.height),\n                "texto": texto_pagina, "blocos": blocos,\n            })\n    if not any(p["texto"].strip() for p in paginas):\n        avisos.append(_aviso("pdf_sem_texto", "Nenhuma página contém texto extraível. A preparação não pode prosseguir sem uma nova fonte textual."))\n    resultado = {\n        "schema": "lingua.gramatica.extracao.v1",\n        "obra": {"arquivo": caminho.name, "sha256_pdf": digest.hexdigest(),\n                 "total_paginas": len(paginas), "metadados": metadados, "metadados_pdf": metadata_pdf},\n        "paginas": paginas, "sumario": sumario,\n        "relatorio": {"avisos": avisos, "ordem_blocos": "declarada_no_pdf",\n                      "normalizacoes": [], "ocr_executado": False,\n                      "separador_blocos": "\\n\\n",\n                      "paginas_com_texto": sum(bool(p["texto"].strip()) for p in paginas),\n                      "caracteres_extraidos": sum(len(b["texto"]) for p in paginas for b in p["blocos"])},\n    }\n    _enriquecer_extracao(resultado)\n    return resultado\n\n\ndef _secoes_por_pagina(extracao):\n    """Usa somente marcadores PDF; páginas com vários títulos são sinalizadas.\n\n    Um marcador não determina em qual bloco da página começa uma seção. Por\n    isso o caminho só é atribuído quando todos os títulos que começam naquela\n    página pertencem à mesma cadeia hierárquica. Em páginas com seções irmãs,\n    retorna apenas o ancestral comum e registra essa limitação.\n    """\n    entradas = sorted(enumerate(extracao.get("sumario", [])), key=lambda x: (x[1]["pagina_pdf"], x[0]))\n    por_pagina = {}\n    for _, item in entradas:\n        por_pagina.setdefault(item["pagina_pdf"], []).append(item)\n    pilha, resultado, avisos = [], {}, []\n    for pagina in extracao["paginas"]:\n        numero = pagina["pagina_pdf"]\n        caminhos = []\n        for item in por_pagina.get(numero, []):\n            nivel = max(1, int(item["nivel"]))\n            pilha = [(n, t) for n, t in pilha if n < nivel]\n            pilha.append((nivel, str(item["titulo"])))\n            caminhos.append([t for _, t in pilha])\n        if len(caminhos) > 1:\n            comum = []\n            for titulos in zip(*caminhos):\n                if len(set(titulos)) != 1:\n                    break\n                comum.append(titulos[0])\n            resultado[numero] = comum\n            avisos.append(_aviso("secoes_na_mesma_pagina", "Vários marcadores começam na página; usa-se apenas o caminho comum, sem atribuir blocos a uma seção incerta.", pagina_pdf=numero))\n        else:\n            resultado[numero] = [t for _, t in pilha]\n    return resultado, avisos\n\n\ndef _contar(tokenizador, texto):\n    return len(tokenizador.encode(texto, add_special_tokens=True, truncation=False))\n\n\ndef _corte_seguro(texto, cabecalho, tokenizador, limite):\n    """Busca um prefixo pelo orçamento real, preferindo fronteiras de texto."""\n    baixo, alto, melhor = 1, len(texto), 0\n    while baixo <= alto:\n        meio = (baixo + alto) // 2\n        if _contar(tokenizador, cabecalho + texto[:meio]) <= limite:\n            melhor, baixo = meio, meio + 1\n        else:\n            alto = meio - 1\n    if melhor == 0:\n        raise ErroPreparacaoLivro("O orçamento de tokens não comporta nem um caractere do texto com o prefixo e os tokens especiais.")\n    if melhor == len(texto):\n        return melhor\n    minimo = max(1, melhor // 2)\n    candidato = None\n    for padrao in (r"[.!?;:]\\s+|\\n+", r"\\s+"):\n        limites = [m.end() for m in re.finditer(padrao, texto[:melhor]) if m.end() >= minimo]\n        if limites:\n            candidato = limites[-1]\n            break\n    if candidato and _contar(tokenizador, cabecalho + texto[:candidato]) <= limite:\n        return candidato\n    return melhor\n\n\ndef _fragmentar_literal(extracao, tokenizador, alvo_tokens=380, max_tokens=512, prefixo="passage: "):\n    """Cria recortes sem truncar texto e com origem para cada trecho citado.\n\n    Os limites incluem prefixo, eventual caminho de seção e tokens especiais.\n    ``tokenizador`` deve ser o mesmo AutoTokenizer usado pelo E5; nenhum modelo\n    é carregado por este módulo. O título da seção pode ser omitido do embedding\n    se não couber, mas continua integralmente nos metadados do fragmento.\n    """\n    if any(isinstance(x, bool) or not isinstance(x, int) for x in (alvo_tokens, max_tokens)):\n        raise ErroPreparacaoLivro("Os limites de tokens devem ser inteiros.")\n    if not 1 <= alvo_tokens <= max_tokens <= 512:\n        raise ErroPreparacaoLivro("Use 1 <= alvo_tokens <= max_tokens <= 512 para o E5.")\n    if not isinstance(prefixo, str):\n        raise ErroPreparacaoLivro("O prefixo deve ser textual.")\n    if not isinstance(extracao, dict) or not isinstance(extracao.get("paginas"), list):\n        raise ErroPreparacaoLivro("A extração não contém uma lista de páginas.")\n    if _contar(tokenizador, prefixo) >= max_tokens:\n        raise ErroPreparacaoLivro("O prefixo e os tokens especiais esgotam o limite.")\n    secoes, avisos = _secoes_por_pagina(extracao)\n    fragmentos, atual, origem, secao_atual = [], "", [], None\n    cabecalho_atual = prefixo\n    caracteres_origem, caracteres_ignorados, blocos_ignorados = 0, 0, []\n    sha_pdf = extracao.get("obra", {}).get("sha256_pdf", "")\n    if not isinstance(sha_pdf, str) or len(sha_pdf) != 64:\n        raise ErroPreparacaoLivro("A extração deve preservar o SHA-256 do PDF.")\n\n    def finalizar():\n        nonlocal atual, origem\n        if not origem:\n            return\n        texto_embedding = cabecalho_atual + atual\n        tokens = _contar(tokenizador, texto_embedding)\n        if not atual.strip() or tokens > max_tokens:\n            raise ErroPreparacaoLivro("Fragmento vazio ou acima do orçamento de tokens.")\n        identidade = json.dumps({"pdf": sha_pdf, "origem": origem, "texto": atual}, ensure_ascii=False, sort_keys=True)\n        fragmentos.append({\n            "id": "frag-" + _sha256(identidade), "linha_vetor": len(fragmentos),\n            "texto": atual, "texto_embedding": texto_embedding, "tokens_embedding": tokens,\n            "secao": list(secao_atual or []), "origem": origem,\n            "sha256_texto": _sha256(atual), "sha256_texto_embedding": _sha256(texto_embedding),\n            "anterior_id": None, "seguinte_id": None,\n        })\n        atual, origem = "", []\n\n    for pagina in extracao["paginas"]:\n        secao = secoes[pagina["pagina_pdf"]]\n        cabecalho = prefixo + (" > ".join(secao) + "\\n\\n" if secao else "")\n        if _contar(tokenizador, cabecalho) >= max_tokens - 8:\n            cabecalho = prefixo\n            avisos.append(_aviso("titulo_fora_embedding", "O caminho da seção não cabe com folga no orçamento e foi mantido apenas nos metadados.", pagina_pdf=pagina["pagina_pdf"]))\n        if secao_atual != secao or cabecalho_atual != cabecalho:\n            finalizar()\n            secao_atual, cabecalho_atual = list(secao), cabecalho\n        for bloco in pagina.get("blocos", []):\n            texto = bloco["texto"]\n            if not isinstance(texto, str):\n                raise ErroPreparacaoLivro("Um bloco da extração não contém texto.")\n            if not texto.strip():\n                caracteres_ignorados += len(texto)\n                blocos_ignorados.append(bloco["id"])\n                continue\n            inicio_pagina = bloco["offset_inicio"]\n            fim_pagina = bloco["offset_fim"]\n            if pagina["texto"][inicio_pagina:fim_pagina] != texto:\n                raise ErroPreparacaoLivro("Os offsets de origem não coincidem com o texto da página.")\n            restantes, inicio = texto, 0\n            while restantes:\n                separador = "\\n\\n" if atual else ""\n                if atual and _contar(tokenizador, cabecalho_atual + atual + separador + restantes) > alvo_tokens:\n                    finalizar()\n                    separador = ""\n                if _contar(tokenizador, cabecalho_atual + restantes) <= max_tokens:\n                    tamanho = len(restantes)\n                else:\n                    # Uma seção muito longa não deve impedir a citação do texto.\n                    if cabecalho_atual != prefixo and _contar(tokenizador, cabecalho_atual + restantes[:1]) > alvo_tokens:\n                        finalizar()\n                        cabecalho_atual = prefixo\n                        avisos.append(_aviso("titulo_fora_embedding", "Título omitido do embedding para comportar o texto sem truncamento.", pagina_pdf=pagina["pagina_pdf"]))\n                    limite_corte = max(alvo_tokens, _contar(tokenizador, cabecalho_atual + restantes[:1]))\n                    if limite_corte > max_tokens:\n                        raise ErroPreparacaoLivro("Um caractere do texto não cabe no orçamento de tokens.")\n                    tamanho = _corte_seguro(restantes, cabecalho_atual, tokenizador, limite_corte)\n                parte = restantes[:tamanho]\n                candidato = atual + separador + parte\n                if _contar(tokenizador, cabecalho_atual + candidato) > max_tokens:\n                    finalizar()\n                    candidato = parte\n                atual = candidato\n                origem.append({\n                    "bloco_id": bloco["id"], "pagina_pdf": pagina["pagina_pdf"],\n                    "pagina_rotulo": pagina.get("pagina_rotulo"), "bbox": list(bloco["bbox"]),\n                    "offset_inicio": inicio_pagina + inicio, "offset_fim": inicio_pagina + inicio + tamanho,\n                    "offset_bloco_inicio": inicio, "offset_bloco_fim": inicio + tamanho,\n                })\n                caracteres_origem += tamanho\n                inicio += tamanho\n                restantes = restantes[tamanho:]\n                if restantes:\n                    finalizar()\n    finalizar()\n    if not fragmentos:\n        raise ErroPreparacaoLivro("O PDF não contém texto utilizável para vetorizar. Confira o relatório de extração.")\n    for indice, fragmento in enumerate(fragmentos):\n        fragmento["anterior_id"] = fragmentos[indice - 1]["id"] if indice else None\n        fragmento["seguinte_id"] = fragmentos[indice + 1]["id"] if indice + 1 < len(fragmentos) else None\n    return {"fragmentos": fragmentos, "relatorio": {\n        "total_fragmentos": len(fragmentos), "alvo_tokens": alvo_tokens, "max_tokens": max_tokens,\n        "max_tokens_observado": max(f["tokens_embedding"] for f in fragmentos),\n        "prefixo": prefixo, "caracteres_origem": caracteres_origem,\n        "caracteres_brancos_ignorados": caracteres_ignorados, "blocos_brancos_ignorados": blocos_ignorados,\n        "normalizacoes": [], "truncamento": False, "avisos": avisos,\n        "separador_blocos": "\\n\\n",\n        "escopo_secoes": "caminho_comum_dos_marcadores_pdf_por_pagina",\n        "revisao_humana_necessaria": True,\n    }}\n\n# A hifenização ortográfica não pode ser distinguida de uma quebra tipográfica\n# apenas pela presença de \'-\'. A lista aprovada é deliberadamente explícita.\n_HIFEN_QUEBRA = re.compile(r"(?P<esquerda>[^\\W\\d_]+)(?P<hifen>[-\\u00ad])(?P<quebra>[ \\t]*\\r?\\n[ \\t\\r\\n]*)(?P<direita>[^\\W\\d_]+)", re.UNICODE)\n\n\ndef _chave_texto(texto):\n    return re.sub(r"\\s+", " ", texto).strip().casefold()\n\n\ndef _origem_bloco(pagina, bloco):\n    return {\n        "bloco_id": bloco["id"], "pagina_pdf": pagina["pagina_pdf"],\n        "pagina_rotulo": pagina.get("pagina_rotulo"), "bbox": list(bloco["bbox"]),\n        "offset_inicio": bloco["offset_inicio"], "offset_fim": bloco["offset_fim"],\n        "offset_bloco_inicio": 0, "offset_bloco_fim": len(bloco["texto"]),\n    }\n\n\ndef _recortar_mapa(mapa, inicio, fim, deslocamento=0):\n    """Recorta intervalos preparados; normalizações remetem ao span raw inteiro."""\n    resultado = []\n    for parte in mapa:\n        a, b = max(inicio, parte["inicio"]), min(fim, parte["fim"])\n        if a >= b:\n            continue\n        fontes = copy.deepcopy(parte["origens"])\n        if parte.get("literal", False):\n            for fonte in fontes:\n                corte_a, corte_b = a - parte["inicio"], parte["fim"] - b\n                fonte["offset_inicio"] += corte_a\n                fonte["offset_bloco_inicio"] += corte_a\n                fonte["offset_fim"] -= corte_b\n                fonte["offset_bloco_fim"] -= corte_b\n        resultado.append({"inicio": a - inicio + deslocamento, "fim": b - inicio + deslocamento,\n                          "origens": fontes, "literal": parte.get("literal", False),\n                          **({"normalizacao_id": parte["normalizacao_id"]} if "normalizacao_id" in parte else {})})\n    return resultado\n\n\ndef _fontes_mapa(mapa):\n    fontes = []\n    for parte in mapa:\n        for fonte in parte["origens"]:\n            if fontes and all(fontes[-1].get(k) == fonte.get(k) for k in ("bloco_id", "pagina_pdf")):\n                anterior = fontes[-1]\n                if fonte["offset_inicio"] <= anterior["offset_fim"]:\n                    anterior["offset_fim"] = max(anterior["offset_fim"], fonte["offset_fim"])\n                    anterior["offset_bloco_fim"] = max(anterior["offset_bloco_fim"], fonte["offset_bloco_fim"])\n                    continue\n            fontes.append(copy.deepcopy(fonte))\n    return fontes\n\n\ndef _detectar_margens(extracao):\n    grupos = {}\n    for pagina in extracao["paginas"]:\n        altura = pagina.get("altura", 0)\n        if not _numero(altura) or altura <= 0:\n            continue\n        for bloco in pagina.get("blocos", []):\n            texto = bloco["texto"].strip()\n            caixa = bloco["bbox"]\n            posicao = "cabecalho" if caixa[3] <= altura * .12 else "rodape" if caixa[1] >= altura * .88 else None\n            if not posicao or not texto or len(texto) > 180 or texto.count("\\n") > 2:\n                continue\n            assinatura = re.sub(r"\\d+", "#", _chave_texto(texto))\n            registro = grupos.setdefault((posicao, assinatura), {\n                "grupo_id": "margem-" + _sha256(posicao + ":" + assinatura)[:16],\n                "assinatura": assinatura, "posicao": posicao, "ids_blocos": [],\n                "paginas": [], "ocorrencias": 0, "amostras": [],\n            })\n            registro["ids_blocos"].append(bloco["id"])\n            registro["paginas"].append(pagina["pagina_pdf"])\n            registro["ocorrencias"] += 1\n            if len(registro["amostras"]) < 3:\n                registro["amostras"].append({"texto": texto, "pagina_pdf": pagina["pagina_pdf"], "bbox": caixa})\n    return [g for g in grupos.values() if len(set(g["paginas"])) >= 3]\n\n\ndef _candidatos_hifenizacao(unidades):\n    candidatos = []\n    for unidade in unidades:\n        for achado in _HIFEN_QUEBRA.finditer(unidade["texto"]):\n            fontes = _fontes_mapa(_recortar_mapa(unidade["mapa"], achado.start(), achado.end()))\n            candidatos.append({\n                "palavra_recomposta": achado["esquerda"] + achado["direita"], "antes": achado.group(),\n                "paginas": list(dict.fromkeys(f["pagina_pdf"] for f in fontes)),\n                "ids_blocos": list(dict.fromkeys(f["bloco_id"] for f in fontes)),\n                "tipo": "hifen_condicional" if achado["hifen"] == "\\u00ad" else "hifen_ambiguo",\n            })\n    return candidatos\n\n\ndef _enriquecer_extracao(extracao):\n    """Sinaliza padrões, sem promover candidatos a exclusões ou seções."""\n    relatorio = extracao.setdefault("relatorio", {"avisos": []})\n    relatorio["candidatos_margem"] = _detectar_margens(extracao)\n    # A listagem também considera palavras partidas entre blocos/páginas.\n    mapa, texto = [], ""\n    for pagina in extracao["paginas"]:\n        for bloco in pagina.get("blocos", []):\n            if texto:\n                texto += "\\n\\n"\n            inicio = len(texto)\n            texto += bloco["texto"]\n            mapa.append({"inicio": inicio, "fim": len(texto), "origens": [_origem_bloco(pagina, bloco)], "literal": True})\n    relatorio["candidatos_hifenizacao"] = _candidatos_hifenizacao([{"texto": texto, "mapa": mapa}])\n    return extracao\n\n\ndef preparar_extracao(extracao, paginas_incluir=None, ids_excluir=(), recompor_palavras=()):\n    """Cria uma camada derivada; a extração literal nunca é sobrescrita.\n\n    Margens são apenas candidatas: só ``ids_excluir`` as retira da busca. Hífen\n    ASCII exige palavra resultante explicitamente aprovada; hífen condicional\n    U+00AD seguido de quebra pode ser recomposto sem inventar grafia. Páginas\n    omitidas, títulos sem conteúdo e cada alteração ficam no relatório.\n    """\n    if not isinstance(extracao, dict) or not isinstance(extracao.get("paginas"), list):\n        raise ErroPreparacaoLivro("A extração não contém páginas.")\n    resultado = copy.deepcopy(extracao)\n    _enriquecer_extracao(resultado)\n    numeros = {p["pagina_pdf"] for p in resultado["paginas"]}\n    selecionadas = numeros if paginas_incluir is None else set(paginas_incluir)\n    if not selecionadas or not selecionadas <= numeros or any(type(n) is not int for n in selecionadas):\n        raise ErroPreparacaoLivro("Selecione páginas PDF existentes; a numeração começa em 1.")\n    ids_conhecidos = {b["id"] for p in resultado["paginas"] for b in p.get("blocos", [])}\n    excluidos = set(ids_excluir)\n    if not excluidos <= ids_conhecidos:\n        raise ErroPreparacaoLivro("Há IDs de exclusão que não pertencem à extração.")\n    if isinstance(recompor_palavras, str) or any(not isinstance(p, str) or not p.isalpha() for p in recompor_palavras):\n        raise ErroPreparacaoLivro("recompor_palavras deve ser uma lista de palavras sem hífen, revisadas pelo usuário.")\n    aprovadas = {p.casefold() for p in recompor_palavras}\n    secoes, avisos = _secoes_por_pagina(resultado)\n    paginas_layout_ambiguo = {a.get("pagina_pdf") for a in resultado.get("relatorio", {}).get("avisos", [])\n                             if a.get("codigo") in ("ordem_leitura_ambigua", "possivel_quadro_ou_tabela")}\n    titulos_pdf = {_chave_texto(s["titulo"]) for s in resultado.get("sumario", [])}\n    tamanhos = [b.get("estilo", {}).get("tamanho_mediano") for p in resultado["paginas"] for b in p.get("blocos", [])\n                if b.get("estilo", {}).get("tamanho_mediano") and len(b["texto"].strip()) >= 120]\n    corpo = statistics.median(tamanhos) if tamanhos else None\n    unidades, exclusoes = [], []\n    for pagina in resultado["paginas"]:\n        for bloco in pagina.get("blocos", []):\n            texto = bloco.get("texto")\n            if not isinstance(texto, str) or pagina["texto"][bloco["offset_inicio"]:bloco["offset_fim"]] != texto:\n                raise ErroPreparacaoLivro("Os offsets da extração não coincidem com o texto original.")\n            fonte = _origem_bloco(pagina, bloco)\n            motivo = "pagina_fora_intervalo" if pagina["pagina_pdf"] not in selecionadas else "exclusao_confirmada" if bloco["id"] in excluidos else "bloco_branco" if not texto.strip() else None\n            if motivo:\n                exclusoes.append({"motivo": motivo, "origem": fonte, "texto": texto})\n                continue\n            estilo = bloco.get("estilo", {})\n            curto = len(texto.strip()) <= 180 and len(texto.strip().splitlines()) <= 3\n            titulo = curto and (_chave_texto(texto) in titulos_pdf or (\n                corpo is not None and pagina["pagina_pdf"] not in paginas_layout_ambiguo\n                and estilo.get("tamanho_mediano", 0) >= corpo * 1.15\n                and not re.search(r"[.!?;:,]\\s*$", texto)))\n            papel = "titulo_candidato" if titulo else "conteudo"\n            secao = secoes[pagina["pagina_pdf"]]\n            juntar = bool(unidades and unidades[-1]["secao"] == secao and (\n                not unidades[-1]["tem_conteudo"]\n                or re.search(r"[^\\W\\d_][-\\u00ad]\\s*$", unidades[-1]["texto"], re.UNICODE)\n                or unidades[-1]["texto"].rstrip().endswith(":")\n                or re.search(r"\\b(?:exemplo|período|oração|frase)\\b[^.!?]*,\\s*$", unidades[-1]["texto"], re.IGNORECASE)))\n            if juntar:\n                unidade = unidades[-1]\n                unidade["texto"] += "\\n\\n"\n            else:\n                unidade = {"id": "unidade-" + bloco["id"], "texto": "", "mapa": [], "secao": list(secao),\n                           "pagina_pdf": pagina["pagina_pdf"], "tem_conteudo": False, "papeis_blocos": {}}\n                unidades.append(unidade)\n            inicio = len(unidade["texto"])\n            unidade["texto"] += texto\n            unidade["mapa"].append({"inicio": inicio, "fim": len(unidade["texto"]), "origens": [fonte], "literal": True})\n            unidade["papeis_blocos"][bloco["id"]] = papel\n            unidade["tem_conteudo"] |= not titulo\n    candidatos = _candidatos_hifenizacao(unidades)\n    normalizacoes = []\n    for unidade in unidades:\n        texto, mapa, cursor, novo, novo_mapa = unidade["texto"], unidade["mapa"], 0, "", []\n        for achado in _HIFEN_QUEBRA.finditer(texto):\n            palavra = achado["esquerda"] + achado["direita"]\n            if palavra.casefold() not in aprovadas and achado["hifen"] != "\\u00ad":\n                continue\n            novo_mapa += _recortar_mapa(mapa, cursor, achado.start(), len(novo))\n            novo += texto[cursor:achado.start()]\n            fontes = _fontes_mapa(_recortar_mapa(mapa, achado.start(), achado.end()))\n            registro = {"id": "normalizacao-" + _sha256(json.dumps([fontes, achado.group(), palavra], sort_keys=True, ensure_ascii=False))[:20],\n                        "tipo": "recomposicao_hifen", "antes": achado.group(), "depois": palavra,\n                        "origem": fontes, "criterio": "hifen_condicional" if achado["hifen"] == "\\u00ad" else "palavra_confirmada"}\n            normalizacoes.append(registro)\n            novo_mapa.append({"inicio": len(novo), "fim": len(novo) + len(palavra), "origens": fontes,\n                              "literal": False, "normalizacao_id": registro["id"]})\n            novo += palavra\n            cursor = achado.end()\n        novo_mapa += _recortar_mapa(mapa, cursor, len(texto), len(novo))\n        novo += texto[cursor:]\n        unidade["texto"], unidade["mapa"] = novo, novo_mapa\n    sem_conteudo = [u for u in unidades if not u["tem_conteudo"]]\n    unidades = [u for u in unidades if u["tem_conteudo"]]\n    paginas_utilizaveis = {f["pagina_pdf"] for u in unidades for parte in u["mapa"] for f in parte["origens"]}\n    for numero in sorted(selecionadas - paginas_utilizaveis):\n        avisos.append(_aviso("pagina_sem_conteudo_apos_preparacao", "Página selecionada sem conteúdo explicativo após a preparação; conferir a fonte e as exclusões antes de vetorizar.", pagina_pdf=numero))\n    if sem_conteudo:\n        avisos.append(_aviso("titulos_sem_conteudo", "Títulos sem texto explicativo foram preservados na preparação, sem gerar vetores isolados.", total=len(sem_conteudo)))\n    resultado["preparacao"] = {\n        "versao": "1.0.0", "unidades": unidades, "exclusoes": exclusoes,\n        "normalizacoes": normalizacoes, "titulos_sem_conteudo": sem_conteudo,\n        "paginas_incluidas": sorted(selecionadas), "candidatos_hifenizacao": candidatos,\n        "recompor_palavras_confirmadas": sorted(aprovadas), "avisos": avisos,\n    }\n    return resultado\n\n\ndef fragmentar_livro(extracao, tokenizador, alvo_tokens=380, max_tokens=512, prefixo="passage: "):\n    """Fragmenta extração literal ou preparada, sem truncamento e com fontes raw.\n\n    Na preparação, cada fragmento leva o mapa de intervalos e o texto bruto.\n    O agrupamento de título/regra/exemplo é preferido, mas o limite real do E5\n    prevalece. Divisões de uma unidade são sinalizadas e ligadas por vizinhos.\n    """\n    if not isinstance(extracao, dict) or "preparacao" not in extracao:\n        return _fragmentar_literal(extracao, tokenizador, alvo_tokens, max_tokens, prefixo)\n    preparacao = extracao["preparacao"]\n    unidades = {u["id"]: u for u in preparacao["unidades"]}\n    if not unidades:\n        raise ErroPreparacaoLivro("Não há conteúdo explicativo para vetorizar depois da preparação. Confira títulos, exclusões e páginas sem texto.")\n    derivada = {"obra": extracao["obra"], "sumario": extracao.get("sumario", []), "paginas": []}\n    for pagina in extracao["paginas"]:\n        blocos, partes, cursor = [], [], 0\n        for unidade in preparacao["unidades"]:\n            if unidade["pagina_pdf"] != pagina["pagina_pdf"]:\n                continue\n            if partes:\n                partes.append("\\n\\n")\n                cursor += 2\n            texto = unidade["texto"]\n            fontes = _fontes_mapa(unidade["mapa"])\n            blocos.append({"id": unidade["id"], "texto": texto, "bbox": fontes[0]["bbox"],\n                           "offset_inicio": cursor, "offset_fim": cursor + len(texto)})\n            partes.append(texto)\n            cursor += len(texto)\n        derivada["paginas"].append({**pagina, "blocos": blocos, "texto": "".join(partes)})\n    resultado = _fragmentar_literal(derivada, tokenizador, alvo_tokens, max_tokens, prefixo)\n    paginas = {p["pagina_pdf"]: p for p in extracao["paginas"]}\n    por_normalizacao = {n["id"]: n for n in preparacao["normalizacoes"]}\n    usos_unidades, mantidos, titulos_isolados = {}, [], []\n    for fragmento in resultado["fragmentos"]:\n        mapa, cursor, papeis = [], 0, {}\n        for origem in fragmento["origem"]:\n            if mapa:\n                cursor += 2\n            unidade = unidades[origem["bloco_id"]]\n            inicio, fim = origem["offset_bloco_inicio"], origem["offset_bloco_fim"]\n            partes = _recortar_mapa(unidade["mapa"], inicio, fim, cursor)\n            mapa.extend(partes)\n            papeis.update(unidade["papeis_blocos"])\n            cursor += fim - inicio\n            usos_unidades.setdefault(unidade["id"], []).append(fragmento["id"])\n        fontes = _fontes_mapa(mapa)\n        if fontes and all(papeis[f["bloco_id"]] == "titulo_candidato" for f in fontes):\n            titulos_isolados.append({"texto": fragmento["texto"], "origem": fontes, "motivo": "titulo_excedeu_orcamento_com_conteudo"})\n            continue\n        fragmento["origem"] = fontes\n        fragmento["mapa_texto_preparado"] = mapa\n        fragmento["texto_extracao_bruta"] = "\\n\\n".join(paginas[f["pagina_pdf"]]["texto"][f["offset_inicio"]:f["offset_fim"]] for f in fontes)\n        fragmento["normalizacoes"] = [copy.deepcopy(por_normalizacao[n]) for n in dict.fromkeys(p["normalizacao_id"] for p in mapa if "normalizacao_id" in p)]\n        # A identidade inclui as fontes reais, e não IDs da camada derivada.\n        identidade = json.dumps({"pdf": extracao["obra"]["sha256_pdf"], "origem": fontes, "texto": fragmento["texto"]}, ensure_ascii=False, sort_keys=True)\n        fragmento["id"] = "frag-" + _sha256(identidade)\n        mantidos.append(fragmento)\n    if not mantidos:\n        raise ErroPreparacaoLivro("Não restou conteúdo explicativo dentro do orçamento E5; confira títulos e extração.")\n    for indice, fragmento in enumerate(mantidos):\n        fragmento["linha_vetor"] = indice\n        fragmento["anterior_id"] = mantidos[indice - 1]["id"] if indice else None\n        fragmento["seguinte_id"] = mantidos[indice + 1]["id"] if indice + 1 < len(mantidos) else None\n    resultado["fragmentos"] = mantidos\n    relatorio = resultado["relatorio"]\n    cobertura = {}\n    for fragmento in mantidos:\n        for fonte in fragmento["origem"]:\n            cobertura.setdefault(fonte["bloco_id"], []).append((fonte["offset_inicio"], fonte["offset_fim"]))\n    caracteres_raw = 0\n    for intervalos in cobertura.values():\n        inicio, fim = sorted(intervalos)[0]\n        for a, b in sorted(intervalos)[1:]:\n            if a <= fim:\n                fim = max(fim, b)\n            else:\n                caracteres_raw += fim - inicio\n                inicio, fim = a, b\n        caracteres_raw += fim - inicio\n    relatorio["caracteres_preparados"] = relatorio["caracteres_origem"]\n    relatorio["caracteres_origem"] = caracteres_raw\n    relatorio.update({"total_fragmentos": len(mantidos), "normalizacoes": copy.deepcopy(preparacao["normalizacoes"]),\n                      "preparacao_versao": preparacao["versao"], "exclusoes": copy.deepcopy(preparacao["exclusoes"]),\n                      "titulos_sem_conteudo": copy.deepcopy(preparacao["titulos_sem_conteudo"]),\n                      "titulos_isolados_por_limite": titulos_isolados,\n                      "paginas_incluidas": preparacao["paginas_incluidas"],\n                      "candidatos_hifenizacao": copy.deepcopy(preparacao["candidatos_hifenizacao"])})\n    relatorio["avisos"].extend(copy.deepcopy(preparacao["avisos"]))\n    divididas = [u for u, usos in usos_unidades.items() if len(usos) > 1]\n    if divididas:\n        relatorio["avisos"].append(_aviso("unidades_divididas_por_limite", "Unidades título/regra/exemplo excederam o orçamento E5; conferir junto com fragmentos vizinhos.", unidades=divididas))\n    return resultado\n', 'gramatica_acervo.py': '"""Vetorização retomável e pacote portátil de uma gramática, sem carregar modelos.\n\nO chamador prepara o PDF, os fragmentos e o SentenceTransformer. Este módulo\nrecebe esse objeto explicitamente; não baixa pesos nem depende do aplicativo.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport zipfile\n\nimport numpy as np\n\n\nMODELO = "intfloat/multilingual-e5-large"\nREVISAO = "3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3"\nDIMENSAO = 1024\nSCHEMA = "gramatica.acervo.v1"\n_SHA = re.compile(r"[0-9a-f]{64}\\Z")\n_ARQUIVOS = (\n    "obra.json", "perfil_embeddings.json", "fragmentos.jsonl", "vetores.npy",\n    "relatorio_extracao.json", "LEIA-ME.md",\n)\n_ANEXOS_OPCIONAIS = (\n    "avaliacao_recuperacao.json", "consultas_teste.json", "vetores_consultas.npy",\n    "paginas.jsonl", "blocos.jsonl",\n    "config_notebook.json", "extracao.json", "preparacao.json",\n    "auditoria_fragmentos.json", "historico_execucao.jsonl",\n)\n\n\nclass ErroAcervo(ValueError):\n    """Entrada incompatível, checkpoint corrompido ou acervo incompleto."""\n\n\ndef _json_bytes(valor):\n    return (json.dumps(valor, ensure_ascii=False, sort_keys=True,\n                       separators=(",", ":"), allow_nan=False) + "\\n").encode("utf-8")\n\n\ndef _ler_json(caminho):\n    def invalido(valor):\n        raise ErroAcervo(f"Constante JSON inválida: {valor}")\n    return json.loads(Path(caminho).read_text(encoding="utf-8"), parse_constant=invalido)\n\n\ndef _sha_bytes(valor):\n    return hashlib.sha256(valor).hexdigest()\n\n\ndef _sha_arquivo(caminho):\n    h = hashlib.sha256()\n    with Path(caminho).open("rb") as arquivo:\n        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):\n            h.update(bloco)\n    return h.hexdigest()\n\n\ndef _atomico(caminho, dados):\n    caminho = Path(caminho)\n    temporario = caminho.with_name(caminho.name + ".tmp")\n    with temporario.open("wb") as arquivo:\n        arquivo.write(dados)\n        arquivo.flush()\n        os.fsync(arquivo.fileno())\n    temporario.replace(caminho)\n\n\ndef _limite_tokens(perfil):\n    if not isinstance(perfil, dict):\n        raise ErroAcervo("O perfil de embeddings deve ser um objeto.")\n    limite = perfil.get("limite_tokens", 512)\n    if type(limite) is not int or not 16 <= limite <= 512:\n        raise ErroAcervo("limite_tokens deve ser um inteiro entre 16 e 512.")\n    return limite\n\n\ndef _perfil(perfil, fragmentos_sha256):\n    perfil = dict(perfil)\n    limite = _limite_tokens(perfil)\n    padroes = {\n        "modelo": MODELO, "revisao": REVISAO,\n        "tokenizador": {"identificacao": MODELO, "revisao": REVISAO},\n        "pooling": "mean", "precisao_inferencia": "float32",\n        "prefixo": "passage: ", "dimensao": DIMENSAO,\n        "normalizacao": "l2", "formato_armazenamento": "float32",\n    }\n    for chave, valor in padroes.items():\n        if chave in perfil and perfil[chave] != valor:\n            raise ErroAcervo(f"Perfil E5 incompatível: {chave}.")\n        perfil[chave] = valor\n    perfil["limite_tokens"] = limite\n    if not _SHA.fullmatch(str(perfil.get("pdf_sha256", ""))):\n        raise ErroAcervo("Informe pdf_sha256: SHA256 do PDF de origem.")\n    if ("fragmentos_sha256" in perfil\n            and perfil["fragmentos_sha256"] != fragmentos_sha256):\n        raise ErroAcervo("O perfil pertence a outros fragmentos.")\n    perfil["fragmentos_sha256"] = fragmentos_sha256\n    hash_informado = perfil.pop("perfil_sha256", None)\n    perfil["perfil_sha256"] = _sha_bytes(_json_bytes(perfil))\n    if hash_informado is not None and hash_informado != perfil["perfil_sha256"]:\n        raise ErroAcervo("O hash do perfil não confere.")\n    return perfil\n\n\ndef _conferir_fragmentos(fragmentos, limite_tokens=512):\n    if not isinstance(fragmentos, list) or not fragmentos:\n        raise ErroAcervo("Não há fragmentos para vetorizar.")\n    ids = set()\n    linhas = []\n    for numero, fragmento in enumerate(fragmentos):\n        if not isinstance(fragmento, dict):\n            raise ErroAcervo(f"Fragmento {numero}: objeto inválido.")\n        identificador = fragmento.get("id")\n        if not isinstance(identificador, str) or not identificador or identificador in ids:\n            raise ErroAcervo(f"Fragmento {numero}: id ausente ou repetido.")\n        ids.add(identificador)\n        if type(fragmento.get("linha_vetor")) is not int or fragmento["linha_vetor"] != numero:\n            raise ErroAcervo(f"Fragmento {numero}: linha_vetor deve seguir a ordem 0..N-1.")\n        texto = fragmento.get("texto")\n        entrada = fragmento.get("texto_embedding")\n        if not isinstance(texto, str) or not texto.strip():\n            raise ErroAcervo(f"Fragmento {numero}: texto vazio.")\n        if not isinstance(entrada, str) or not entrada.startswith("passage: "):\n            raise ErroAcervo(f"Fragmento {numero}: texto_embedding sem prefixo passage.")\n        tokens = fragmento.get("tokens_embedding")\n        if type(tokens) is not int or not 1 <= tokens <= limite_tokens:\n            raise ErroAcervo(f"Fragmento {numero}: contagem de tokens inválida.")\n        for chave, conteudo in (("sha256_texto", texto), ("sha256_texto_embedding", entrada)):\n            if fragmento.get(chave) != _sha_bytes(conteudo.encode("utf-8")):\n                raise ErroAcervo(f"Fragmento {numero}: {chave} não confere.")\n        linhas.append(_json_bytes(fragmento))\n    return b"".join(linhas)\n\n\ndef _conferir_matriz(matriz, linhas):\n    if matriz.shape != (linhas, DIMENSAO) or matriz.dtype != np.dtype("float32"):\n        raise ErroAcervo(f"Matriz deve ser float32 com dimensão ({linhas}, {DIMENSAO}).")\n    for inicio in range(0, linhas, 1024):\n        bloco = matriz[inicio:inicio + 1024]\n        if not np.isfinite(bloco).all():\n            raise ErroAcervo("Matriz contém NaN ou infinito.")\n        normas = np.linalg.norm(bloco.astype(np.float64), axis=1)\n        if not np.allclose(normas, 1.0, atol=2e-5, rtol=2e-5):\n            raise ErroAcervo("Matriz contém vetores sem normalização L2 válida.")\n\n\ndef _validar_modelo(modelo, fragmentos, limite_perfil=512):\n    dimensao = getattr(modelo, "get_sentence_embedding_dimension", None)\n    if callable(dimensao) and dimensao() != DIMENSAO:\n        raise ErroAcervo("O modelo carregado não gera 1024 dimensões.")\n    dtype = getattr(modelo, "dtype", None)\n    if dtype is not None and str(dtype) not in ("torch.float32", "float32"):\n        raise ErroAcervo("Carregue os pesos em float32 para corresponder ao perfil.")\n    tokenizador = getattr(modelo, "tokenizer", None)\n    if tokenizador is None or not callable(getattr(tokenizador, "encode", None)):\n        raise ErroAcervo("O modelo deve expor seu tokenizer para conferir ausência de truncamento.")\n    limite = getattr(modelo, "max_seq_length", None)\n    if type(limite) is not int or limite <= 0:\n        raise ErroAcervo("O modelo deve informar max_seq_length.")\n    for fragmento in fragmentos:\n        tokens = len(tokenizador.encode(fragmento["texto_embedding"], add_special_tokens=True,\n                                        truncation=False))\n        if tokens != fragmento["tokens_embedding"] or tokens > min(limite, limite_perfil, 512):\n            raise ErroAcervo(f"Fragmento {fragmento[\'id\']}: tokens incompatíveis ou truncamento possível.")\n\n\ndef _oom(erro):\n    # PyTorch é opcional aqui. A exceção pode ser torch.cuda.OutOfMemoryError,\n    # mas também RuntimeError nas versões suportadas pelo Colab.\n    return (isinstance(erro, RuntimeError)\n            and (type(erro).__name__ == "OutOfMemoryError"\n                 or "cuda out of memory" in str(erro).lower()\n                 or "cuda error: out of memory" in str(erro).lower()))\n\n\ndef _limpar_cuda(modelo):\n    # Somente o objeto de modelo já carregado pode optar por essa limpeza.\n    limpar = getattr(modelo, "limpar_cache_cuda", None)\n    if callable(limpar):\n        limpar()\n    # SentenceTransformer não fornece esse método. Importação tardia somente\n    # após OOM, e somente se o chamador realmente carregou torch.\n    import sys\n    torch = sys.modules.get("torch")\n    if torch is not None and getattr(torch, "cuda", None) is not None:\n        torch.cuda.empty_cache()\n\n\ndef _ler_lotes(pasta, perfil, total):\n    lotes = []\n    proximo = 0\n    for metadados_path in sorted(pasta.glob("lote-*.json")):\n        dados = _ler_json(metadados_path)\n        inicio, fim = dados.get("inicio"), dados.get("fim")\n        if (dados.get("perfil_sha256") != perfil["perfil_sha256"]\n                or type(inicio) is not int or type(fim) is not int\n                or inicio != proximo or not inicio < fim <= total):\n            raise ErroAcervo("Checkpoint pertence a outra execução ou contém lacunas.")\n        esperado = f"lote-{inicio:09d}-{fim:09d}.npy"\n        if dados.get("arquivo") != esperado or metadados_path.stem != esperado[:-4]:\n            raise ErroAcervo("Nome de checkpoint inválido.")\n        arquivo = pasta / esperado\n        if not arquivo.is_file() or _sha_arquivo(arquivo) != dados.get("sha256"):\n            raise ErroAcervo("SHA256 do checkpoint não confere; não será reutilizado.")\n        matriz = np.load(arquivo, mmap_mode="r", allow_pickle=False)\n        _conferir_matriz(matriz, fim - inicio)\n        lotes.append((inicio, fim, arquivo))\n        proximo = fim\n    return lotes, proximo\n\n\ndef vetorizar_fragmentos(fragmentos, modelo, pasta, perfil, lote_inicial=64, progresso=None):\n    """Gera vetores reais pelo modelo recebido, com retomada estritamente compatível.\n\n    ``progresso``, quando presente, recebe um dicionário com ``concluidos``,\n    ``total``, ``lote`` e ``retomado``. Erros interrompem a execução; os lotes\n    válidos permanecem disponíveis. Nenhum vetor sintético substitui uma falha.\n    """\n    if type(lote_inicial) is not int or lote_inicial < 1:\n        raise ErroAcervo("lote_inicial deve ser um inteiro positivo.")\n    bytes_fragmentos = _conferir_fragmentos(fragmentos, _limite_tokens(perfil))\n    perfil = _perfil(perfil, _sha_bytes(bytes_fragmentos))\n    pasta = Path(pasta)\n    pasta.mkdir(parents=True, exist_ok=True)\n    checkpoints = pasta / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    perfil_path = pasta / "perfil_embeddings.json"\n    if perfil_path.exists() and _perfil(_ler_json(perfil_path), _sha_bytes(bytes_fragmentos)) != perfil:\n        raise ErroAcervo("Esta pasta contém outro perfil ou outros fragmentos. Use outra pasta.")\n    fragmentos_path = pasta / "fragmentos.jsonl"\n    if fragmentos_path.exists() and fragmentos_path.read_bytes() != bytes_fragmentos:\n        raise ErroAcervo("Esta pasta contém outros fragmentos. Use outra pasta.")\n    checkpoint_perfil = checkpoints / "perfil.json"\n    if checkpoint_perfil.exists() and _ler_json(checkpoint_perfil) != perfil:\n        raise ErroAcervo("O perfil dos checkpoints não confere.")\n    _validar_modelo(modelo, fragmentos, perfil["limite_tokens"])\n    _atomico(perfil_path, _json_bytes(perfil))\n    _atomico(fragmentos_path, bytes_fragmentos)\n    _atomico(checkpoint_perfil, _json_bytes(perfil))\n    lotes, concluido = _ler_lotes(checkpoints, perfil, len(fragmentos))\n    estado_path = pasta / "estado_vetorizacao.json"\n    _atomico(estado_path, _json_bytes({"estado": "incompleto", "perfil_sha256": perfil["perfil_sha256"]}))\n    tamanho = lote_inicial\n    if progresso:\n        progresso({"concluidos": concluido, "total": len(fragmentos), "lote": tamanho,\n                   "retomado": concluido > 0})\n    while concluido < len(fragmentos):\n        fim = min(concluido + tamanho, len(fragmentos))\n        entradas = [f["texto_embedding"] for f in fragmentos[concluido:fim]]\n        try:\n            vetores = modelo.encode(entradas, batch_size=tamanho, prompt="",\n                                    normalize_embeddings=True, convert_to_numpy=True,\n                                    show_progress_bar=False)\n        except RuntimeError as erro:\n            if not _oom(erro) or tamanho == 1:\n                raise\n            _limpar_cuda(modelo)\n            tamanho = max(1, tamanho // 2)\n            continue\n        vetores = np.asarray(vetores)\n        # Armazenar fp32 não promete precisão de inferência fp32: o notebook\n        # também deve carregar os pesos e executar o modelo em float32.\n        if not np.issubdtype(vetores.dtype, np.floating):\n            raise ErroAcervo("O modelo retornou valores que não são ponto flutuante.")\n        vetores = np.asarray(vetores, dtype=np.float32)\n        _conferir_matriz(vetores, fim - concluido)\n        nome = f"lote-{concluido:09d}-{fim:09d}.npy"\n        caminho = checkpoints / nome\n        temporario = caminho.with_suffix(".npy.tmp")\n        with temporario.open("wb") as arquivo:\n            np.save(arquivo, vetores, allow_pickle=False)\n            arquivo.flush()\n            os.fsync(arquivo.fileno())\n        temporario.replace(caminho)\n        _atomico(caminho.with_suffix(".json"), _json_bytes({\n            "inicio": concluido, "fim": fim, "arquivo": nome,\n            "sha256": _sha_arquivo(caminho), "perfil_sha256": perfil["perfil_sha256"],\n        }))\n        lotes.append((concluido, fim, caminho))\n        concluido = fim\n        if progresso:\n            progresso({"concluidos": concluido, "total": len(fragmentos), "lote": tamanho,\n                       "retomado": False})\n    destino = pasta / "vetores.npy"\n    temporario = pasta / "vetores.npy.tmp"\n    matriz = np.lib.format.open_memmap(temporario, mode="w+", dtype=np.float32,\n                                       shape=(len(fragmentos), DIMENSAO))\n    for inicio, fim, caminho in lotes:\n        matriz[inicio:fim] = np.load(caminho, mmap_mode="r", allow_pickle=False)\n    matriz.flush()\n    _conferir_matriz(matriz, len(fragmentos))\n    del matriz\n    temporario.replace(destino)\n    _atomico(estado_path, _json_bytes({\n        "estado": "completo", "perfil_sha256": perfil["perfil_sha256"],\n        "fragmentos": len(fragmentos), "vetores_sha256": _sha_arquivo(destino),\n    }))\n    return destino\n\n\ndef validar_acervo(pasta):\n    """Confere os arquivos locais, referências e a matriz; nunca faz inferência."""\n    pasta = Path(pasta)\n    erros = []\n    resultado = {"schema": SCHEMA, "valido": False, "estado": "incompleto", "erros": erros}\n    try:\n        for nome in _ARQUIVOS[:-1]:\n            if not (pasta / nome).is_file() or (pasta / nome).is_symlink():\n                raise ErroAcervo(f"Arquivo obrigatório ausente ou inválido: {nome}.")\n        fragmentos = [json.loads(linha) for linha in (pasta / "fragmentos.jsonl").read_text(\n            encoding="utf-8").splitlines() if linha.strip()]\n        perfil_bruto = _ler_json(pasta / "perfil_embeddings.json")\n        canonico = _conferir_fragmentos(fragmentos, _limite_tokens(perfil_bruto))\n        if canonico != (pasta / "fragmentos.jsonl").read_bytes():\n            raise ErroAcervo("fragmentos.jsonl não coincide com a serialização canônica.")\n        perfil = _perfil(perfil_bruto, _sha_bytes(canonico))\n        obra = _ler_json(pasta / "obra.json")\n        if not isinstance(obra, dict) or not str(obra.get("titulo", "")).strip():\n            raise ErroAcervo("obra.json precisa identificar o título da obra.")\n        if obra.get("pdf_sha256") != perfil["pdf_sha256"]:\n            raise ErroAcervo("SHA256 do PDF diverge entre obra e perfil.")\n        if not isinstance(_ler_json(pasta / "relatorio_extracao.json"), dict):\n            raise ErroAcervo("Relatório de extração inválido.")\n        matriz = np.load(pasta / "vetores.npy", mmap_mode="r", allow_pickle=False)\n        _conferir_matriz(matriz, len(fragmentos))\n        estado = _ler_json(pasta / "estado_vetorizacao.json")\n        if (estado.get("estado") != "completo" or estado.get("perfil_sha256") != perfil["perfil_sha256"]\n                or estado.get("fragmentos") != len(fragmentos)\n                or estado.get("vetores_sha256") != _sha_arquivo(pasta / "vetores.npy")):\n            raise ErroAcervo("Execução não concluída ou integridade final inválida.")\n        resultado.update(valido=True, estado="completo", fragmentos=len(fragmentos),\n                         dimensao=DIMENSAO, dtype="float32", perfil_sha256=perfil["perfil_sha256"])\n    except (OSError, ValueError, TypeError, KeyError) as erro:\n        erros.append(f"{type(erro).__name__}: {erro}")\n    return resultado\n\n\ndef exportar_acervo(pasta, destino=None):\n    """Exporta apenas acervo completo; checkpoints e pesos ficam fora do ZIP."""\n    pasta = Path(pasta)\n    relatorio = validar_acervo(pasta)\n    if not relatorio["valido"]:\n        raise ErroAcervo("O acervo está incompleto: " + "; ".join(relatorio["erros"]))\n    leia_me = (\n        "# Acervo gramatical vetorizado\\n\\n"\n        "Os textos e suas referências de origem acompanham vetores E5 reais. "\n        "Este pacote não aplica regras gramaticais automaticamente e não é o formato "\n        "de vetorização de relatos do Agente Analista. A importação no Língua será "\n        "uma etapa separada.\\n\\n"\n        "`fragmentos.jsonl`: linha_vetor identifica a linha correspondente em "\n        "`vetores.npy`. `texto` preserva a citação; `texto_embedding` registra "\n        "exatamente a entrada enviada ao modelo, incluindo prefixo e seção. "\n        "Os vetores são float32, 1024 dimensões, com normalização L2.\\n\\n"\n        "`obra.json` identifica o PDF por SHA256; páginas PDF e rótulos impressos "\n        "são distintos. Confira amostras, notas e leitura de colunas no relatório "\n        "de extração. Integridade numérica não demonstra fidelidade editorial "\n        "nem correção de uma futura análise linguística.\\n\\n"\n        "`manifesto.json` registra tamanho e SHA256 de cada arquivo, exceto o próprio "\n        "manifesto. Pesos do modelo, PDF e checkpoints não são incluídos.\\n"\n    )\n    _atomico(pasta / "LEIA-ME.md", leia_me.encode("utf-8"))\n    arquivos = list(_ARQUIVOS) + ["estado_vetorizacao.json"]\n    for nome in _ANEXOS_OPCIONAIS:\n        if (pasta / nome).is_file():\n            arquivos.append(nome)\n    codigo = pasta / "codigo"\n    if codigo.is_dir() and not codigo.is_symlink():\n        arquivos.extend("codigo/" + f.name for f in sorted(codigo.glob("*.py"))\n                        if f.is_file() and not f.is_symlink())\n    for nome in arquivos:\n        if (pasta / nome).is_symlink():\n            raise ErroAcervo(f"Não exportar link simbólico: {nome}.")\n    manifesto = {\n        "schema": SCHEMA, "estado": "completo", "validacao": relatorio,\n        "arquivos": {nome: {"sha256": _sha_arquivo(pasta / nome),\n                            "bytes": (pasta / nome).stat().st_size} for nome in arquivos},\n    }\n    _atomico(pasta / "manifesto.json", _json_bytes(manifesto))\n    destino = Path(destino) if destino else pasta.with_name(pasta.name + ".zip")\n    destino.parent.mkdir(parents=True, exist_ok=True)\n    if destino.resolve() in {(pasta / nome).resolve() for nome in arquivos + ["manifesto.json"]}:\n        raise ErroAcervo("O ZIP de destino não pode substituir arquivos do acervo.")\n    if destino.resolve().is_relative_to((pasta / "checkpoints").resolve()):\n        raise ErroAcervo("O destino não pode ficar na pasta de checkpoints.")\n    temporario = destino.with_name(destino.name + ".tmp")\n    try:\n        with zipfile.ZipFile(temporario, "w", compression=zipfile.ZIP_DEFLATED,\n                             compresslevel=6, allowZip64=True) as pacote:\n            for nome in arquivos + ["manifesto.json"]:\n                pacote.write(pasta / nome, arcname=nome)\n        temporario.replace(destino)\n    finally:\n        if temporario.exists():\n            temporario.unlink()\n    return destino\n'}

PASTA_CODIGO = Path("/content/lingua_gramatica_codigo")
PASTA_CODIGO.mkdir(parents=True, exist_ok=True)
for nome, conteudo in FONTES.items():
    (PASTA_CODIGO / nome).write_text(conteudo, encoding="utf-8")
HASHES_FONTES = {nome: hashlib.sha256(texto.encode()).hexdigest() for nome, texto in FONTES.items()}
print("Fontes incorporadas ao notebook:", HASHES_FONTES)

## 2. Preparar o ambiente

O ambiente isolado reutiliza o PyTorch CUDA já instalado pelo Colab.
Ele é criado sem `ensurepip`; o pip do Python do Colab instala os pacotes
no ambiente de destino. Reexecutar a célula também repara a configuração
de um ambiente parcial deixado por uma instalação interrompida.
As bibliotecas de extração e embeddings são instaladas nele, sem substituir
o PyTorch da GPU. O download inicial do E5 Large é de aproximadamente 2,2 GB.
As versões efetivamente usadas serão registradas no pacote.

In [ ]:
import os
import subprocess
import sys
import textwrap
import venv
from pathlib import Path

PASTA_AMBIENTE = Path("/content/venv_gramatica_livro")
PYTHON_LIVRO = PASTA_AMBIENTE / "bin" / "python"
# Sempre reconfigura: uma falha de ensurepip pode deixar o Python criado,
# mas sem acesso aos pacotes do Colab. Não remove PDF ou checkpoints.
venv.EnvBuilder(with_pip=False, system_site_packages=True).create(PASTA_AMBIENTE)
PIP_LIVRO = [sys.executable, "-m", "pip", "--python", str(PYTHON_LIVRO)]
subprocess.run(PIP_LIVRO + ["--version"], check=True)
subprocess.run([str(PYTHON_LIVRO), "-c",
    "import sys, torch; from pathlib import Path; "
    "assert Path(sys.prefix).resolve() == Path(sys.argv[1]).resolve(), 'Ambiente Python incorreto.'; "
    "assert torch.cuda.is_available(), 'Selecione GPU no ambiente de execução do Colab.'; "
    "print('PyTorch CUDA existente:', torch.__version__); "
    "print('GPU:', torch.cuda.get_device_name(0)); "
    "print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))",
    str(PASTA_AMBIENTE)], check=True)
subprocess.run(PIP_LIVRO + ["install", "--disable-pip-version-check",
    "--no-input", "PyMuPDF==1.26.7", "sentence-transformers==5.2.0",
    "transformers==4.57.3", "huggingface-hub==0.36.0", "numpy>=1.26,<3",
    "sentencepiece==0.2.1", "protobuf>=5.29,<7", "rank-bm25==0.2.2"], check=True)
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
subprocess.run([str(PYTHON_LIVRO), '-c',
    'import torch, numpy, pymupdf; from transformers import AutoTokenizer; '
    'from sentence_transformers import SentenceTransformer; from rank_bm25 import BM25Okapi; '
    'assert torch.cuda.is_available(); print("OK: bibliotecas e CUDA importadas.")'], check=True)

def executar(codigo):
    cabecalho = (
        "from pathlib import Path\nimport json, sys, hashlib\n"
        "config = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))\n"
        "pasta = Path(config['pasta_execucao'])\n"
    )
    subprocess.run([str(PYTHON_LIVRO), "-c", cabecalho + textwrap.dedent(codigo),
        str(CONFIG_PATH)], cwd=PASTA_CODIGO, check=True)

## 3. Escolher o PDF

Use um PDF com texto selecionável. A extração preserva o texto e aponta
páginas vazias, possíveis colunas e outras limitações; não aplica OCR.
Se usar `CAMINHO_PDF`, o arquivo deve estar acessível no Colab ou no Drive.

In [ ]:
if CAMINHO_PDF:
    PDF = Path(CAMINHO_PDF).expanduser()
else:
    from google.colab import files
    enviados = files.upload()
    if len(enviados) != 1:
        raise ValueError("Escolha exatamente um arquivo PDF.")
    nome, dados = next(iter(enviados.items()))
    PDF = Path("/content") / Path(nome).name
    PDF.write_bytes(dados)
    del enviados, dados
if not PDF.is_file() or PDF.suffix.lower() != ".pdf":
    raise ValueError("O caminho precisa apontar para um PDF existente.")
digest = hashlib.sha256()
with PDF.open("rb") as arquivo:
    for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
        digest.update(bloco)
PDF_SHA256 = digest.hexdigest()
PDF_CUNHA_RECONHECIDO = PDF_SHA256 == 'bd25df1ccaca3320d0ab7a62ad550b5943df092ee65c7ac3173d5f2daa53ac3f'
PAGINAS_SELECIONADAS = PAGINAS_INCLUIR
if PAGINAS_SELECIONADAS is None and USAR_CORPO_CUNHA_RECONHECIDO and PDF_CUNHA_RECONHECIDO:
    PAGINAS_SELECIONADAS = list(range(41, 771))
    print('PDF Cunha/Cintra reconhecido: corpo 41–770; demais páginas registradas como excluídas.')
elif PAGINAS_SELECIONADAS is None:
    print('Serão consideradas todas as páginas. Confira os dados bibliográficos e a cobertura.')
parametros = {"modelo": 'intfloat/multilingual-e5-large', "revisao": '3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3',
    "prefixo": "passage: ", "alvo_tokens": ALVO_TOKENS, "limite_tokens": LIMITE_TOKENS,
    "hashes_fontes": HASHES_FONTES,
    "metadados": {"titulo": TITULO or PDF.stem, "autor": AUTOR, "edicao": EDICAO, "ano": ANO}}
chave = hashlib.sha256(json.dumps(parametros, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:16]
PASTA_EXECUCAO = PASTA_BASE / PDF_SHA256[:16] / chave
PASTA_EXECUCAO.mkdir(parents=True, exist_ok=True)
PASTA_INSPECAO = PASTA_EXECUCAO
CONFIG_PATH = PASTA_EXECUCAO / "config_notebook.json"
config = {**parametros, "pdf": str(PDF.resolve()), "pdf_sha256": PDF_SHA256,
    "pasta_execucao": str(PASTA_EXECUCAO), "lote_inicial": LOTE_INICIAL,
    "cache_modelos": "/content/cache_e5_gramatica"}
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
CONFIG_BASE = dict(config)
print("PDF:", PDF.name, "SHA-256:", PDF_SHA256)
print("Pasta de trabalho/retomada:", PASTA_EXECUCAO)

In [ ]:
executar(r'''
from gramatica_pdf import extrair_pdf
extraido = extrair_pdf(config['pdf'], metadados=config['metadados'])
(pasta / 'extracao.json').write_text(json.dumps(extraido, ensure_ascii=False, indent=2), encoding='utf-8')
with (pasta / 'paginas.jsonl').open('w', encoding='utf-8') as arquivo:
    for pagina in extraido['paginas']:
        arquivo.write(json.dumps(pagina, ensure_ascii=False) + '\n')
print('Páginas:', len(extraido['paginas']))
print('Relatório:', json.dumps(extraido['relatorio'], ensure_ascii=False, indent=2))
''')

## 4. Conferir a extração antes dos vetores

Confira páginas e imagens, especialmente exemplos, tabelas e palavras
partidas. No PDF já examinado, as páginas 54 e 55 tiveram extração pobre.
Os candidatos a cabeçalho/rodapé são sugestões; somente os grupos que você
confirmar serão excluídos do texto preparado. O original permanece no pacote.
A paginação impressa não é inferida automaticamente pela página do PDF.

In [ ]:
extraido = json.loads((PASTA_EXECUCAO / "extracao.json").read_text(encoding="utf-8"))
selecionadas = PAGINAS_SELECIONADAS or list(range(1, len(extraido['paginas']) + 1))
PAGINAS_PARA_CONFERIR = ([54, 55, 97, 98, 658, 659] if PDF_CUNHA_RECONHECIDO else
    [selecionadas[0], selecionadas[len(selecionadas) // 2], selecionadas[-1]])
for numero in dict.fromkeys(PAGINAS_PARA_CONFERIR):
    pagina = extraido["paginas"][numero - 1]
    print("\nPÁGINA DO PDF", numero, "— rótulo:", pagina.get("pagina_rotulo"))
    print(pagina["texto"][:3500])
print("\nExtração integral:", PASTA_EXECUCAO / "extracao.json")
print("Confira outras páginas e exemplos antes de continuar.")

In [ ]:
MOSTRAR_IMAGENS_PDF = True
if MOSTRAR_IMAGENS_PDF:
    (PASTA_INSPECAO / 'paginas_conferencia.json').write_text(json.dumps(PAGINAS_PARA_CONFERIR), encoding='utf-8')
    executar(r'''
    import pymupdf
    saida = pasta / 'imagens_conferencia'
    saida.mkdir(exist_ok=True)
    with pymupdf.open(config['pdf']) as pdf:
        for numero in json.loads((pasta / 'paginas_conferencia.json').read_text()):
            if not 1 <= numero <= len(pdf):
                raise ValueError('Página de conferência fora do PDF.')
            pdf[numero - 1].get_pixmap(dpi=90).save(saida / f'pagina-{numero}.png')
    ''')
    from IPython.display import Image, display
    for numero in dict.fromkeys(PAGINAS_PARA_CONFERIR):
        print('Página PDF', numero)
        display(Image(filename=str(PASTA_INSPECAO / 'imagens_conferencia' / f'pagina-{numero}.png')))

In [ ]:
grupos = extraido['relatorio'].get('candidatos_margem', [])
for grupo in grupos:
    print(grupo['grupo_id'], '|', grupo['posicao'], '| ocorrências:', grupo['ocorrencias'],
          '| páginas:', grupo['paginas'][:12])
    print('  Amostras:', json.dumps(grupo['amostras'][:2], ensure_ascii=False))
print('Copie abaixo os IDs dos grupos que são realmente cabeçalhos/rodapés.')
print('Não selecione títulos, notas ou exemplos que pertençam ao conteúdo.')

In [ ]:
GRUPOS_MARGEM_CONFIRMADOS = []  # Ex.: ['margem-...']; copie os IDs mostrados acima.
IDS_BLOCOS_EXCLUIR = []        # Opcional: exclusões individuais explicitamente conferidas.
por_grupo = {g['grupo_id']: g for g in grupos}
if set(GRUPOS_MARGEM_CONFIRMADOS) - por_grupo.keys():
    raise ValueError('ID de grupo desconhecido; use os IDs desta extração.')
ids_excluir = sorted(set(IDS_BLOCOS_EXCLUIR).union(*[
    set(por_grupo[g]['ids_blocos']) for g in GRUPOS_MARGEM_CONFIRMADOS]))
config = {**CONFIG_BASE, 'paginas_incluir': PAGINAS_SELECIONADAS, 'ids_excluir': ids_excluir}
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
executar(r'''
from gramatica_pdf import preparar_extracao
extraido = json.loads((pasta / 'extracao.json').read_text(encoding='utf-8'))
preparado = preparar_extracao(extraido, paginas_incluir=config['paginas_incluir'],
    ids_excluir=config['ids_excluir'])
(pasta / 'preparacao_previa.json').write_text(json.dumps(preparado, ensure_ascii=False), encoding='utf-8')
''')
previa = json.loads((PASTA_INSPECAO / 'preparacao_previa.json').read_text(encoding='utf-8'))
candidatos = previa['preparacao'].get('candidatos_hifenizacao', [])
for candidato in candidatos[:100]:
    print(candidato['palavra_recomposta'], '| páginas:', candidato['paginas'],
          '| antes:', repr(candidato['antes']))
print('Candidatos a junção:', len(candidatos), '(lista integral na preparação prévia).')
print('Confirme somente palavras partidas pela diagramação; preserve compostos e notação gramatical.')

In [ ]:
RECOMPOR_PALAVRAS = []  # Copie formas unidas conferidas: por exemplo 'integrantes'.
import shutil
opcoes = {'paginas_incluir': PAGINAS_SELECIONADAS, 'ids_excluir': ids_excluir,
          'recompor_palavras': sorted(set(RECOMPOR_PALAVRAS))}
assinatura = {k: CONFIG_BASE[k] for k in ('modelo', 'revisao', 'prefixo', 'alvo_tokens',
              'limite_tokens', 'hashes_fontes', 'metadados')}
assinatura['preparacao'] = opcoes
chave = hashlib.sha256(json.dumps(assinatura, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:16]
PASTA_EXECUCAO = PASTA_BASE / PDF_SHA256[:16] / ('exec-' + chave)
PASTA_EXECUCAO.mkdir(parents=True, exist_ok=True)
for nome in ('extracao.json', 'paginas.jsonl'):
    shutil.copyfile(PASTA_INSPECAO / nome, PASTA_EXECUCAO / nome)
config = {**CONFIG_BASE, **opcoes, 'pasta_execucao': str(PASTA_EXECUCAO)}
CONFIG_PATH = PASTA_EXECUCAO / 'config_notebook.json'
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
EXTRACAO_CONFERIDA = False
FRAGMENTOS_CONFERIDOS = False
executar(r'''
from gramatica_pdf import preparar_extracao
extraido = json.loads((pasta / 'extracao.json').read_text(encoding='utf-8'))
preparado = preparar_extracao(extraido, paginas_incluir=config['paginas_incluir'],
    ids_excluir=config['ids_excluir'], recompor_palavras=config['recompor_palavras'])
(pasta / 'preparacao.json').write_text(json.dumps(preparado, ensure_ascii=False), encoding='utf-8')
p = preparado['preparacao']
print('Páginas incluídas:', len(p['paginas_incluidas']))
print('Exclusões registradas:', len(p['exclusoes']), '| normalizações:', len(p['normalizacoes']))
print('Candidatos de hifenização para conferir:', len(p['candidatos_hifenizacao']))
for unidade in p['unidades'][:3]:
    print('\nAMOSTRA PREPARADA:', unidade['texto'][:1600])
print('Preparação integral:', pasta / 'preparacao.json')
''')
print('Pasta dos vetores e checkpoints:', PASTA_EXECUCAO)

Após conferir a extração, altere a célula abaixo para `True` e execute-a.
A célula de vetorização interrompe antes de carregar o E5 se essa conferência
não estiver marcada. Se encontrar erros, preserve o relatório e revise a
extração antes de gerar o acervo.

In [ ]:
EXTRACAO_CONFERIDA = False  # Altere para True depois da conferência.

In [ ]:
if not EXTRACAO_CONFERIDA:
    raise RuntimeError("Confira a extração e marque EXTRACAO_CONFERIDA = True antes de continuar.")
executar(r'''
from transformers import AutoTokenizer
from gramatica_pdf import fragmentar_livro
tokenizador = AutoTokenizer.from_pretrained(config['modelo'], revision=config['revisao'],
    cache_dir=config['cache_modelos'], trust_remote_code=False)
extraido = json.loads((pasta / 'preparacao.json').read_text(encoding='utf-8'))
if extraido['obra']['sha256_pdf'] != config['pdf_sha256']:
    raise ValueError('O PDF extraído não corresponde ao arquivo identificado. Reexecute a preparação.')
resultado = fragmentar_livro(extraido, tokenizador, alvo_tokens=config['alvo_tokens'],
    max_tokens=config['limite_tokens'], prefixo=config['prefixo'])
fragmentos = resultado['fragmentos']
if not fragmentos:
    raise ValueError('Nenhum fragmento foi produzido. Confira a extração.')
(pasta / 'fragmentos_preparados.json').write_text(json.dumps(fragmentos, ensure_ascii=False), encoding='utf-8')
obra = {**extraido['obra'], **config['metadados'], 'pdf_sha256': config['pdf_sha256']}
(pasta / 'obra.json').write_text(json.dumps(obra, ensure_ascii=False, indent=2), encoding='utf-8')
relatorio = {'extracao': extraido['relatorio'], 'fragmentacao': resultado['relatorio'],
    'preparacao': {k: v for k, v in extraido['preparacao'].items() if k != 'unidades'},
    'conferencia_extracao': 'confirmada_no_notebook', 'paginas_impressas': 'nao_conferidas'}
(pasta / 'relatorio_extracao.json').write_text(json.dumps(relatorio, ensure_ascii=False, indent=2), encoding='utf-8')
print('Fragmentos preparados:', len(fragmentos))
from collections import Counter
tokens = [f['tokens_embedding'] for f in fragmentos]
contagens = Counter(f['texto_embedding'] for f in fragmentos)
auditoria = {'fragmentos': len(fragmentos), 'tokens_min': min(tokens), 'tokens_max': max(tokens),
    'fragmentos_menos_32_tokens': sum(n < 32 for n in tokens),
    'entradas_repetidas_excedentes': sum(n - 1 for n in contagens.values()),
    'avisos_fragmentacao': resultado['relatorio'].get('avisos', []),
    'conferencia_fragmentos': 'pendente'}
(pasta / 'auditoria_fragmentos.json').write_text(json.dumps(auditoria, ensure_ascii=False, indent=2), encoding='utf-8')
print('Auditoria:', json.dumps(auditoria, ensure_ascii=False, indent=2)[:6000])
print('Fragmentos mais curtos — confira se há regra ou exemplo, e não só título:')
for item in sorted(fragmentos, key=lambda f: f['tokens_embedding'])[:12]:
    print(item['id'], '|', item['tokens_embedding'], '|', repr(item['texto'][:300]), '|', item['origem'])
for item in fragmentos[:3]:
    print('\n', item['id'], '| tokens:', item['tokens_embedding'])
    print(item['texto_embedding'][:1400])
''')

Confira os fragmentos mais curtos e alguns trechos de adjetivas, concessivas
e morfologia em `fragmentos_preparados.json`. Os títulos reconhecidos são
agrupados com conteúdo; layouts ambíguos continuam exigindo conferência.
Verifique também se regra, exemplo e ressalva precisam de um vizinho.
Após conferir, mude a célula abaixo para `True`. Alterar páginas ou correções
cria outra pasta de execução, preservando os vetores anteriores.

In [ ]:
FRAGMENTOS_CONFERIDOS = False  # Altere para True após conferir os recortes.

## 5. Gerar os vetores E5 na GPU

Entradas já incluem `passage: ` e são conferidas sem truncamento.
A inferência é float32, com mean pooling e saída L2 de 1.024 dimensões.
Falta de memória reduz o lote; outros erros interrompem sem publicar um
pacote completo. Reexecutar esta célula retoma lotes válidos na mesma pasta.

In [ ]:
if not EXTRACAO_CONFERIDA or not FRAGMENTOS_CONFERIDOS:
    raise RuntimeError("Confira extração e fragmentos antes de carregar o E5.")
executar(r'''
import importlib.metadata as metadata
import platform
import datetime
import torch
from sentence_transformers import SentenceTransformer
from gramatica_acervo import vetorizar_fragmentos
if not torch.cuda.is_available():
    raise RuntimeError('GPU CUDA indisponível. Selecione GPU no ambiente do Colab.')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
modelo = SentenceTransformer(config['modelo'], revision=config['revisao'], device='cuda',
    cache_folder=config['cache_modelos'], trust_remote_code=False)
modelo.float().eval()
modelo.max_seq_length = config['limite_tokens']
if modelo.get_sentence_embedding_dimension() != 1024:
    raise ValueError('O modelo não produziu o espaço E5 de 1024 dimensões.')
pooling = [m.get_config_dict() for m in modelo if type(m).__name__ == 'Pooling']
if len(pooling) != 1 or not pooling[0].get('pooling_mode_mean_tokens') or any(
    valor for chave, valor in pooling[0].items()
    if chave.startswith('pooling_mode_') and chave != 'pooling_mode_mean_tokens'):
    raise ValueError('Pooling do modelo incompatível com o perfil de referência.')
ambiente = {'python': platform.python_version(), 'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'bibliotecas': {n: metadata.version(n) for n in
        ('sentence-transformers', 'transformers', 'huggingface-hub', 'numpy', 'PyMuPDF', 'sentencepiece')}}
tentativa = {'inicio_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'ambiente': ambiente, 'gpu': torch.cuda.get_device_name(0),
    'vram_bytes': torch.cuda.get_device_properties(0).total_memory,
    'conferencia_extracao': True, 'conferencia_fragmentos': True}
with (pasta / 'historico_execucao.jsonl').open('a', encoding='utf-8') as arquivo:
    arquivo.write(json.dumps(tentativa, ensure_ascii=False) + '\n')
perfil = {'modelo': config['modelo'], 'revisao': config['revisao'],
    'tokenizador': {'identificacao': config['modelo'], 'revisao': config['revisao']},
    'pooling': 'mean', 'pooling_configuracao': pooling, 'prefixo': config['prefixo'],
    'dimensao': 1024, 'normalizacao': 'l2', 'formato_armazenamento': 'float32',
    'precisao_inferencia': 'float32', 'limite_tokens': config['limite_tokens'],
    'pdf_sha256': config['pdf_sha256'], 'hashes_fontes': config['hashes_fontes'],
    'ambiente': ambiente, 'tf32': False}
fragmentos = json.loads((pasta / 'fragmentos_preparados.json').read_text(encoding='utf-8'))
auditoria_path = pasta / 'auditoria_fragmentos.json'
auditoria = json.loads(auditoria_path.read_text(encoding='utf-8'))
auditoria['conferencia_fragmentos'] = 'confirmada_no_notebook'
auditoria_path.write_text(json.dumps(auditoria, ensure_ascii=False, indent=2), encoding='utf-8')
matriz = vetorizar_fragmentos(fragmentos, modelo, pasta, perfil,
    lote_inicial=config['lote_inicial'], progresso=lambda evento: print(json.dumps(evento), flush=True))
print('Matriz concluída:', matriz)
print('Ambiente:', json.dumps(ambiente, ensure_ascii=False, indent=2))
''')

## 6. Inspecionar a recuperação (opcional)

Estas consultas ajudam a conferir o acervo. Não são uma avaliação de acurácia.
Ajuste os assuntos ao livro e confira as passagens e suas páginas. Esta célula
gera consultas com `query: ` na GPU e combina E5 com busca lexical BM25.
Resultados repetidos são agrupados, e vizinhos da mesma seção acompanham
a passagem para inspeção. O acervo permanece independente do aplicativo.

In [ ]:
CONSULTAS_TESTE = ['oração subordinada adjetiva restritiva e explicativa',
                  'oração adverbial concessiva introduzida por embora',
                  'objeto direto e objeto indireto',
                  'modo subjuntivo e flexão verbal',
                  'pronomes pessoais átonos e colocação pronominal']
(PASTA_EXECUCAO / 'consultas_teste.json').write_text(json.dumps(CONSULTAS_TESTE, ensure_ascii=False), encoding='utf-8')
executar(r'''
import numpy as np
import re
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from gramatica_acervo import validar_acervo
validacao = validar_acervo(pasta)
if not validacao['valido']:
    raise ValueError(validacao)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
modelo = SentenceTransformer(config['modelo'], revision=config['revisao'], device='cuda',
    cache_folder=config['cache_modelos'], trust_remote_code=False)
modelo.float().eval()
modelo.max_seq_length = config['limite_tokens']
consultas = json.loads((pasta / 'consultas_teste.json').read_text(encoding='utf-8'))
entradas = ['query: ' + texto for texto in consultas]
if any(len(modelo.tokenizer.encode(t, add_special_tokens=True, truncation=False)) > config['limite_tokens'] for t in entradas):
    raise ValueError('Consulta de inspeção longa demais; reduza o texto sem truncar silenciosamente.')
vetores = modelo.encode(entradas, prompt='', normalize_embeddings=True, convert_to_numpy=True,
    show_progress_bar=False, batch_size=8)
np.save(pasta / 'vetores_consultas.npy', vetores.astype(np.float32), allow_pickle=False)
matriz = np.load(pasta / 'vetores.npy', mmap_mode='r', allow_pickle=False)
fragmentos = [json.loads(linha) for linha in (pasta / 'fragmentos.jsonl').read_text(encoding='utf-8').splitlines()]
def termos(texto):
    return re.findall(r'[^\W_]+', texto.casefold())
bm25 = BM25Okapi([termos(f['texto']) for f in fragmentos])
resultados = []
for consulta, vetor in zip(consultas, vetores):
    scores = matriz @ vetor.astype(np.float32)
    scores_lexicais = np.asarray(bm25.get_scores(termos(consulta)))
    rankings = [np.argsort(-scores, kind='stable')[:32]]
    if np.any(scores_lexicais != 0):
        rankings.append(np.argsort(-scores_lexicais, kind='stable')[:32])
    fusao = {}
    for ranking in rankings:
        for posicao, i in enumerate(ranking, start=1):
            fusao[int(i)] = fusao.get(int(i), 0.0) + 1 / (60 + posicao)
    indices = sorted(fusao, key=lambda i: (-fusao[i], -float(scores[i]), i))
    itens, vistos = [], set()
    for i in indices:
        f = fragmentos[i]
        chave = ' '.join(f['texto'].casefold().split())
        if chave in vistos:
            continue
        vistos.add(chave)
        vizinhos = [{'fragmento_id': fragmentos[j]['id'], 'texto': fragmentos[j]['texto'],
            'origem': fragmentos[j]['origem']} for j in (i - 1, i + 1)
            if 0 <= j < len(fragmentos) and fragmentos[j]['secao'] == f['secao']]
        itens.append({'fragmento_id': f['id'], 'linha_vetor': i,
            'similaridade_cosseno': float(scores[i]), 'score_rrf': fusao[i],
            'score_bm25': float(scores_lexicais[i]),
            'texto': f['texto'], 'origem': f['origem'], 'vizinhos': vizinhos})
        if len(itens) == 5:
            break
    resultados.append({'consulta': consulta, 'resultados': itens})
    print('\nCONSULTA:', consulta)
    for item in itens:
        print(item['score_rrf'], item['fragmento_id'], item['texto'][:900], 'Origem:', item['origem'])
(pasta / 'avaliacao_recuperacao.json').write_text(json.dumps({
    'tipo': 'inspecao_sem_referencia_manual',
    'limite': 'Similaridade não é confiança gramatical nem acurácia.',
    'consultas': resultados}, ensure_ascii=False, indent=2), encoding='utf-8')
''')

## 7. Conferir e baixar o pacote

O pacote inclui textos e suas origens, vetores, perfil, relatório de extração,
estado da execução e manifesto de hashes. PDF, pesos e checkpoints ficam fora
do ZIP. Rótulos de páginas não são certificação de paginação impressa.
Guarde o ZIP; ele será usado na futura integração do acervo no aplicativo.

In [ ]:
executar(r'''
import shutil
from gramatica_acervo import validar_acervo, exportar_acervo
validacao = validar_acervo(pasta)
if not validacao['valido']:
    raise ValueError(validacao)
print(json.dumps(validacao, ensure_ascii=False, indent=2))
fontes = pasta / 'codigo'
fontes.mkdir(exist_ok=True)
for nome in ('gramatica_pdf.py', 'gramatica_acervo.py'):
    shutil.copyfile(nome, fontes / nome)
destino = exportar_acervo(pasta)
(pasta / 'pacote_download.json').write_text(json.dumps({'caminho': str(destino)}), encoding='utf-8')
print('Pacote validado:', destino)
''')

In [ ]:
from google.colab import files
pacote = json.loads((PASTA_EXECUCAO / 'pacote_download.json').read_text(encoding='utf-8'))['caminho']
files.download(pacote)
print('Download solicitado:', pacote)
print('Guarde também o PDF original e mantenha a pasta de trabalho se precisar retomar.')